# 05: PD Model Training

This notebook trains six classification models for Probability of Default (PD) estimation. Each model is tuned via RandomizedSearchCV on a stratified sample of 300,000 observations. The model with the best **cross-validated** AUC is selected, calibrated with isotonic regression on a dedicated calibration split, and only then evaluated on the held-out test set.

The data is split three ways so that no step reuses data it was fitted on:

| Split | Share | Used for |
|---|---|---|
| Train | 70% | hyperparameter search and model fitting |
| Calibration | 10% | fitting the isotonic calibrator |
| Test | 20% | final, untouched evaluation of ranking and calibration |

**Input:** `data/processed/checkpoint_df_PD_training.pkl`, `data/processed/features_PD_num_for_training.pkl`, `data/processed/features_PD_cat_for_training.pkl`  
**Output:** `models/<best_model>_PD.pkl`, `models/PD_model_comparison.csv`, `models/calibrated_model_PD.pkl`, `data/processed/checkpoint_df_master_with_PD.pkl`

## Libraries

In [ ]:
# Run from the repository root so that data/, models/ and src/ resolve the same way
# whether the notebook is opened from the root or from production_notebooks/.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "production_notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import shap
import os

from sklearn.model_selection     import train_test_split
from sklearn.calibration         import CalibratedClassifierCV, calibration_curve
from sklearn.frozen              import FrozenEstimator
from sklearn.metrics             import roc_auc_score, average_precision_score, brier_score_loss

from train_models_PD  import train_all_models_PD
from preprocess       import preprocess_features, apply_preprocessing
from readme_figures   import plot_pd_model_comparison, plot_calibration_curves, display_name

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_df_PD_training.pkl", "rb") as f:
    data = pickle.load(f)
df_PD_clean = data["df_PD_clean"]

with open("data/processed/features_PD_num_for_training.pkl", "rb") as f:
    num_features_PD = pickle.load(f)

with open("data/processed/features_PD_cat_for_training.pkl", "rb") as f:
    cat_features_PD = pickle.load(f)

features_PD = num_features_PD + cat_features_PD

print(f"df_PD_clean shape: {df_PD_clean.shape}")
print(f"Features: {features_PD}")

## 2. Train / Calibration / Test Split

Stratified splits preserving the class distribution. The test set (20%) is set aside first; the calibration set (10% of the total) is then carved out of the remainder. All models are trained on the same training split.

In [ ]:
X = df_PD_clean[features_PD]
y = df_PD_clean["Target_Variable_Default"]

# Test set first (20%), then a calibration set (10% of the total) from the rest
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_rest, y_rest, test_size=0.125, random_state=42, stratify=y_rest
)
del X_rest, y_rest

for name, ys in [("train", y_train), ("calibration", y_cal), ("test", y_test)]:
    print(f"{name:<12}: {len(ys):>9,} rows | default rate {ys.mean():.4f}")

## 3. Preprocessing

RobustScaler fitted on the training split only and applied unchanged to the calibration and test splits. One-Hot Encoding applied to categorical features, with the calibration and test columns aligned to the training schema. Applied uniformly across all models as a portfolio simplification.

In [ ]:
X_train, X_cal, scaler_PD = preprocess_features(
    X_train, X_cal, cat_features_PD, num_features_PD
)
X_test = apply_preprocessing(X_test, cat_features_PD, num_features_PD, X_train.columns, scaler_PD)

print(f"X_train: {X_train.shape} | X_cal: {X_cal.shape} | X_test: {X_test.shape}")

## 4. Model Training

Six classification models are trained:
- Logistic Regression (interpretable baseline)
- Decision Tree
- Random Forest
- XGBoost
- LightGBM
- CatBoost

All models are tuned via RandomizedSearchCV (300k stratified sample, 5-fold CV, optimizing AUC-ROC). Results are saved to `models/`.

In [ ]:
results_PD = train_all_models_PD(
    X_train, y_train,
    X_test,  y_test,
    sample_size=300_000
)

In [ ]:
# Results summary, ranked by cross-validated AUC (the selection criterion).
# Test metrics are shown for reference only; they play no role in the choice.
results_PD

In [ ]:
# Model comparison chart for the README (selected model highlighted)
plot_pd_model_comparison(results_PD, selected=results_PD["cv_auc"].idxmax())
plt.show()

## 5. Probability Calibration

The models are trained with class weighting (`class_weight='balanced'` / `scale_pos_weight`) to learn the minority class, which pushes the raw `predict_proba` outputs well above the observed default rate. Those outputs rank borrowers correctly but cannot be used as probabilities in `EL = PD x LGD x EAD`.

Isotonic regression maps the raw scores to probabilities. Two details matter:

- The calibrator is fitted on the **calibration split**, which the model has never seen. The trained model is wrapped in `FrozenEstimator`, so calibration does not refit it.
- Calibration quality is then measured on the **test split**, which neither the model nor the calibrator has seen. A calibrator evaluated on its own fitting data matches the observed default rate by construction, so that comparison would prove nothing.

In [ ]:
# Load the model with the best cross-validated AUC
best_name = results_PD["cv_auc"].idxmax()
with open(f"models/{best_name}_PD.pkl", "rb") as f:
    model_PD = pickle.load(f)
print(f"Selected model: {best_name} (CV AUC {results_PD.loc[best_name, 'cv_auc']})")

# Raw probabilities on the untouched test split
y_prob_test_raw = model_PD.predict_proba(X_test)[:, 1]
print(f"Mean PD raw (test): {y_prob_test_raw.mean():.4f} | default rate (test): {y_test.mean():.4f}")

In [ ]:
# Fit the isotonic calibrator on the calibration split; FrozenEstimator keeps the
# trained model as it is (no refitting on the calibration data).
calibrated_model_PD = CalibratedClassifierCV(FrozenEstimator(model_PD), method="isotonic")
calibrated_model_PD.fit(X_cal, y_cal)

# Evaluate on the test split, unseen by both the model and the calibrator
y_prob_test_cal = calibrated_model_PD.predict_proba(X_test)[:, 1]
print(f"Mean PD calibrated (test): {y_prob_test_cal.mean():.4f} | default rate (test): {y_test.mean():.4f}")
print(f"Brier score raw: {brier_score_loss(y_test, y_prob_test_raw):.4f} | "
      f"calibrated: {brier_score_loss(y_test, y_prob_test_cal):.4f}")
print(f"AUC raw: {roc_auc_score(y_test, y_prob_test_raw):.4f} | "
      f"calibrated: {roc_auc_score(y_test, y_prob_test_cal):.4f}")

In [ ]:
# Calibration curves on the test split, before and after (also saved for the README)
plot_calibration_curves(y_prob_test_raw, y_prob_test_cal, y_test, display_name(best_name))
plt.show()

In [ ]:
# Save calibrated model
with open("models/calibrated_model_PD.pkl", "wb") as f:
    pickle.dump(calibrated_model_PD, f)

print("Calibrated model saved.")

## 6. SHAP Analysis

In [ ]:
# Full dataset for inference
X_full = pd.concat([X_train, X_cal, X_test], axis=0).sort_index()

# SHAP on a representative sample
X_shap = X_test.sample(n=50_000, random_state=42)

explainer   = shap.TreeExplainer(model_PD)
shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

print(f"SHAP values shape: {shap_values.shape}")

shap.summary_plot(shap_values, X_shap, max_display=X_shap.shape[1], show=False)
plt.savefig("assets/shap_pd_final.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. PD Inference on Full Portfolio

In [ ]:
# Calibrated PD over full dataset
y_prob_PD = calibrated_model_PD.predict_proba(X_full)[:, 1]

print(f"PD computed for {len(y_prob_PD):,} loans")
print(f"Mean PD    : {y_prob_PD.mean():.4f}")
print(f"Median PD  : {np.median(y_prob_PD):.4f}")

In [ ]:
# Verify PD separation between defaulted and non-defaulted populations
with open("data/processed/checkpoint_DataFrame_for_EDA.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

df_master_features["PD"] = pd.Series(y_prob_PD, index=X_full.index)

# Record which split each loan belongs to, so that 07_expected_loss can validate
# the Expected Loss on the test loans alone (out of sample).
df_master_features["split_PD"] = pd.Series("train", index=X_train.index)
df_master_features.loc[X_cal.index, "split_PD"] = "calibration"
df_master_features.loc[X_test.index, "split_PD"] = "test"

print("PD: non-defaulted:")
print(df_master_features.loc[df_master_features["Target_Variable_Default"] == 0, "PD"].describe())
print("\nPD: defaulted:")
print(df_master_features.loc[df_master_features["Target_Variable_Default"] == 1, "PD"].describe())

In [ ]:
# Save df_master_features with PD column
with open("data/processed/checkpoint_df_master_with_PD.pkl", "wb") as f:
    pickle.dump({"df_master_features": df_master_features}, f)

print("Saved df_master_features with PD column.")